# Neutrality Test - Streamlit App

Interactive app for `neutrality_lrt`, with the **bottleneck Nb estimated from the data** per recipient (from `bottleneck_function`).

**Input format:** CSV files where **columns = samples/recipients** and **rows = features**.

**Requirements**
```
pip install streamlit numpy pandas scipy statsmodels matplotlib
```

Run all cells in order; the last cell starts the server. All file writes/reads use UTF-8 (Windows-safe).

## 1 - Install dependencies (run once)

In [1]:
%pip install streamlit numpy pandas scipy statsmodels matplotlib --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 - Write `neutrality_test.py`

In [2]:
neutrality_test_source = '"""\nNeutrality test for donor->recipient transmission experiments.\n"""\nfrom __future__ import annotations\nfrom typing import Optional, Sequence, Tuple, Union\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize_scalar\nfrom scipy.special import gammaln\nfrom scipy.stats import chi2\nfrom statsmodels.stats.multitest import multipletests\n\n_MIN_PVAL = float(np.nextafter(0.0, 1.0))\n\ndef _log_beta(a, b):\n    return float(gammaln(a) + gammaln(b) - gammaln(a + b))\n\ndef _log_bb_pmf(x, n, a, b):\n    return float(\n        (gammaln(n + 1) - gammaln(x + 1) - gammaln(n - x + 1))\n        + _log_beta(x + a, n - x + b)\n        - _log_beta(a, b)\n    )\n\ndef _loglik_feature(q, x_vec, n_vec, Nb_vec, *, alpha0, q_bounds):\n    lo, hi = q_bounds\n    q = float(np.clip(q, lo, hi))\n    ll = 0.0\n    for xm, nm, Nbm in zip(x_vec, n_vec, Nb_vec):\n        if nm <= 0:\n            continue\n        a = float(Nbm * q + alpha0)\n        b = float(Nbm * (1.0 - q) + alpha0)\n        ll += _log_bb_pmf(int(xm), int(nm), a, b)\n    return float(ll)\n\ndef _fit_q_mle(x_vec, n_vec, Nb_vec, *, alpha0, q_bounds):\n    lo, hi = q_bounds\n    def nll(q):\n        return -_loglik_feature(q, x_vec, n_vec, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)\n    res = minimize_scalar(nll, bounds=(lo, hi), method="bounded")\n    return float(res.x), float(-res.fun)\n\ndef _as_1d_float_array(x, length):\n    if np.isscalar(x):\n        return np.full(length, float(x), dtype=float)\n    arr = np.asarray(x, dtype=float).reshape(-1)\n    if arr.shape[0] != length:\n        raise ValueError(f"bottleneck has length {arr.shape[0]}, expected {length}")\n    return arr\n\ndef neutrality_lrt(\n    donor_counts, recipient_counts, bottleneck, *,\n    feature_ids=None, pseudocount=1e-6, alpha0=1e-12,\n    min_p=1e-8, q_bounds=(1e-12, 1 - 1e-12),\n    fdr_alpha=0.05, fdr_method="fdr_bh",\n):\n    if isinstance(recipient_counts, pd.DataFrame):\n        X = recipient_counts.to_numpy(dtype=np.int64)\n        cols_from_recip = list(recipient_counts.columns)\n    else:\n        X = np.asarray(recipient_counts, dtype=np.int64)\n        cols_from_recip = None\n    if X.ndim == 1:\n        X = X.reshape(1, -1)\n    if X.ndim != 2:\n        raise ValueError("recipient_counts must be 1D or 2D")\n    M, K = X.shape\n\n    if isinstance(donor_counts, pd.Series):\n        donor_arr = donor_counts.to_numpy(dtype=float).reshape(-1)\n        ids_from_donor = list(donor_counts.index)\n    else:\n        donor_arr = np.asarray(donor_counts, dtype=float).reshape(-1)\n        ids_from_donor = None\n    if donor_arr.shape[0] != K:\n        raise ValueError(f"donor_counts length does not match K={K}")\n\n    if feature_ids is None:\n        if cols_from_recip is not None:\n            feature_ids = cols_from_recip\n        elif ids_from_donor is not None:\n            feature_ids = ids_from_donor\n        else:\n            feature_ids = list(range(K))\n\n    Nb_vec = _as_1d_float_array(bottleneck, M)\n    n_tot  = X.sum(axis=1).astype(np.int64)\n    donor_adj = donor_arr + float(pseudocount)\n    p_donor   = donor_adj / donor_adj.sum()\n\n    q_hat = np.full(K, np.nan)\n    LR    = np.full(K, np.nan)\n    pval  = np.full(K, np.nan)\n\n    for j in range(K):\n        p0 = float(p_donor[j])\n        if (not np.isfinite(p0)) or (p0 < min_p) or (p0 > 1.0 - min_p):\n            continue\n        x_vec = X[:, j].astype(np.int64, copy=False)\n        ll0 = _loglik_feature(p0, x_vec, n_tot, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)\n        qh, ll1 = _fit_q_mle(x_vec, n_tot, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)\n        q_hat[j] = qh\n        LRj = max(0.0, 2.0 * (ll1 - ll0))\n        LR[j] = LRj\n        p = float(chi2.sf(LRj, df=1))\n        if (not np.isfinite(p)) or p <= 0.0:\n            p = _MIN_PVAL\n        pval[j] = p\n\n    tested = np.isfinite(pval)\n    qval   = np.full(K, np.nan)\n    reject = np.zeros(K, dtype=bool)\n    if tested.any():\n        r, qv, _, _ = multipletests(pval[tested], alpha=float(fdr_alpha), method=str(fdr_method))\n        qval[tested]  = np.maximum(qv, _MIN_PVAL)\n        pval[tested]  = np.maximum(pval[tested], _MIN_PVAL)\n        reject[tested] = r\n\n    res = pd.DataFrame({\n        "feature": list(feature_ids),\n        "p_donor": p_donor,\n        "q_hat_recipient_center": q_hat,\n        "LR": LR,\n        "pval": pval,\n        "qval_FDR": qval,\n        "reject_FDR": reject,\n    })\n    direction = pd.Series([None]*len(res), dtype="object")\n    mask = np.isfinite(res["q_hat_recipient_center"].to_numpy(dtype=float))\n    direction.loc[mask] = np.where(\n        res.loc[mask, "q_hat_recipient_center"] > res.loc[mask, "p_donor"],\n        "up_in_recipient", "down_in_recipient",\n    )\n    res["direction"] = direction\n    return res.sort_values(["pval","LR"], ascending=[True,False]).reset_index(drop=True)\n'
with open('neutrality_test.py', 'w', encoding='utf-8') as f:
    f.write(neutrality_test_source)
print('neutrality_test.py' + ' written.')

neutrality_test.py written.


## 3 - Write `bottleneck_function.py` (Nb estimator)

In [3]:
bottleneck_source = '"""\nBottleneck (effective population size Nb) estimation via Dirichlet-Multinomial MLE.\nExtracted from bottleneck_function.ipynb.\n"""\nimport numpy as np\nfrom scipy.special import gammaln\n\n\ndef donor_frequencies(counts_t0, pseudocount=1e-6):\n    """Convert raw donor counts at time 0 into a probability vector p (with pseudocount)."""\n    c0 = np.asarray(counts_t0, dtype=float)\n    if c0.ndim != 1:\n        raise ValueError("counts_t0 must be a 1D vector.")\n    if not np.isfinite(c0).all() or np.any(c0 < 0):\n        raise ValueError("counts_t0 must be finite and nonnegative.")\n    p = c0 + float(pseudocount)\n    s = float(p.sum())\n    if s <= 0:\n        raise ValueError("sum(counts_t0 + pseudocount) must be > 0.")\n    return p / s\n\n\ndef dm_loglik(counts_t1, Nb, p, alpha0=1e-12):\n    """Dirichlet-multinomial log-likelihood for counts x given Nb and base probabilities p."""\n    x = np.asarray(counts_t1, dtype=np.int64)\n    if x.ndim != 1:\n        raise ValueError("counts_t1 must be a 1D vector.")\n    if np.any(x < 0):\n        raise ValueError("counts_t1 must be nonnegative.")\n    n = int(x.sum())\n    alpha = Nb * np.asarray(p, dtype=float) + float(alpha0)\n    A = float(alpha.sum())\n    return float((gammaln(n + 1) - np.sum(gammaln(x + 1)))\n                 + (gammaln(A) - gammaln(A + n))\n                 + np.sum(gammaln(alpha + x) - gammaln(alpha)))\n\n\ndef bottleneck_mle_from_p(counts_t1, p, nb_max=5_000_000, n_grid=220, refine=220, alpha0=1e-12):\n    """MLE of Nb given recipient counts and donor probability vector p (two-stage grid search)."""\n    x = np.asarray(counts_t1, dtype=np.int64)\n    if x.sum() == 0:\n        return np.nan\n    grid = np.unique(np.round(np.logspace(0, np.log10(nb_max), n_grid)).astype(int))\n    ll = np.array([dm_loglik(x, Nb, p, alpha0=alpha0) for Nb in grid])\n    Nb0 = int(grid[np.argmax(ll)])\n    lo = max(1, Nb0 // 2)\n    hi = min(nb_max, Nb0 * 2)\n    grid2 = np.unique(np.linspace(lo, hi, refine).astype(int))\n    ll2 = np.array([dm_loglik(x, Nb, p, alpha0=alpha0) for Nb in grid2])\n    return int(grid2[np.argmax(ll2)])\n\n\ndef bottleneck_from_two_timepoints(counts_t0, counts_t1,\n                                   donor_pseudocount=1e-6, alpha0=1e-12,\n                                   nb_max=5_000_000, n_grid=220, refine=220,\n                                   return_p=False):\n    """Estimate Nb from raw donor counts (t0) and one recipient\'s raw counts (t1)."""\n    p = donor_frequencies(counts_t0, pseudocount=donor_pseudocount)\n    nb_hat = bottleneck_mle_from_p(counts_t1, p, nb_max=nb_max, n_grid=n_grid, refine=refine, alpha0=alpha0)\n    return (nb_hat, p) if return_p else nb_hat\n'
with open('bottleneck_function.py', 'w', encoding='utf-8') as f:
    f.write(bottleneck_source)
print('bottleneck_function.py' + ' written.')

bottleneck_function.py written.


## 4 - Write the Streamlit app `app_neutrality.py`

Nb is estimated per recipient and fed into the test; there is no manual bottleneck field.

In [4]:
app_source = 'import io\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport streamlit as st\nfrom neutrality_test import neutrality_lrt\nfrom bottleneck_function import bottleneck_from_two_timepoints\n\nst.set_page_config(page_title="Neutrality LRT", layout="wide")\nst.title("Neutrality Test - Donor to Recipient Transmission")\nst.markdown(\n    """\n    Test whether each feature (taxon / gene / sgRNA) is **neutral** during transmission:\n    - **H0**: recipient center frequency = donor frequency\n    - **H1**: recipient center frequency is free\n\n    The bottleneck size **Nb is estimated from the data** for every recipient\n    (Dirichlet-Multinomial MLE from the donor and that recipient\'s counts),\n    so it is not a manual parameter.\n    """\n)\n\nwith st.sidebar:\n    st.header("Parameters")\n    fdr_alpha   = st.slider("FDR alpha", 0.01, 0.20, 0.05, 0.01)\n    pseudocount = st.number_input("Donor pseudocount", value=1e-6, format="%.2e")\n    min_p       = st.number_input("min_p (skip near-zero donor freqs)", value=1e-8, format="%.2e")\n    fdr_method  = st.selectbox("FDR method", ["fdr_bh", "fdr_by", "holm", "bonferroni"])\n    with st.expander("Advanced: Nb estimator"):\n        nb_max = int(st.number_input("nb_max", value=5000000, step=100000))\n        n_grid = int(st.number_input("coarse grid points", value=220, step=20))\n        refine = int(st.number_input("refine points", value=220, step=20))\n    st.markdown("---")\n    st.markdown("**Input format** - CSV files, columns = samples/recipients, rows = features (taxa / genes / sgRNAs).")\n\ntab_upload, tab_demo = st.tabs(["Upload your data", "Run demo"])\n\ndonor_df = None\nrecipient_df = None\n\nwith tab_upload:\n    col1, col2 = st.columns(2)\n    with col1:\n        st.subheader("Donor / inoculum counts")\n        st.caption("One column of raw counts; one row per feature.")\n        donor_file = st.file_uploader("Upload donor CSV", type="csv", key="donor")\n        if donor_file:\n            donor_raw = pd.read_csv(donor_file, index_col=0)\n            st.dataframe(donor_raw.head(), use_container_width=True)\n            donor_df = donor_raw.T\n    with col2:\n        st.subheader("Recipient counts")\n        st.caption("Columns = recipients, rows = features (same feature order as donor).")\n        recip_file = st.file_uploader("Upload recipient CSV", type="csv", key="recip")\n        if recip_file:\n            recip_raw = pd.read_csv(recip_file, index_col=0)\n            st.dataframe(recip_raw.head(), use_container_width=True)\n            recipient_df = recip_raw.T\n\nwith tab_demo:\n    st.markdown(\n        """\n        Generates **synthetic** data: 8 features, 6 recipients.\n        Features A and B are shifted in recipients to simulate non-neutrality.\n        Shown in the documented layout (rows = features, columns = samples).\n        """\n    )\n    if st.button("Generate demo data"):\n        rng = np.random.default_rng(42)\n        K, M = 8, 6\n        features = [f"Feature_{chr(65+i)}" for i in range(K)]\n        donor_counts_demo = rng.integers(50, 500, size=K).astype(float)\n        donor_df = pd.DataFrame([donor_counts_demo], columns=features, index=["donor"])\n        p = donor_counts_demo / donor_counts_demo.sum()\n        p_recip = p.copy()\n        p_recip[0] *= 3; p_recip[1] *= 0.2\n        p_recip /= p_recip.sum()\n        rows = []\n        for _ in range(M):\n            n = rng.integers(800, 1200)\n            rows.append(rng.multinomial(n, p_recip))\n        recipient_df = pd.DataFrame(rows, columns=features, index=[f"recipient_{i+1}" for i in range(M)])\n        st.session_state["donor_df"] = donor_df\n        st.session_state["recipient_df"] = recipient_df\n        st.success("Demo data generated!")\n        c1, c2 = st.columns(2)\n        with c1:\n            st.write("**Donor** (rows = features)"); st.dataframe(donor_df.T, use_container_width=True)\n        with c2:\n            st.write("**Recipients** (rows = features)"); st.dataframe(recipient_df.T, use_container_width=True)\n\nif donor_df is None and "donor_df" in st.session_state:\n    donor_df = st.session_state["donor_df"]\nif recipient_df is None and "recipient_df" in st.session_state:\n    recipient_df = st.session_state["recipient_df"]\n\nif donor_df is not None and recipient_df is not None:\n    st.markdown("---")\n    if st.button("Estimate Nb and run neutrality LRT", type="primary"):\n        if donor_df.shape[0] == 1:\n            donor_series = donor_df.iloc[0]\n        elif donor_df.shape[1] == 1:\n            donor_series = donor_df.iloc[:, 0]\n        else:\n            donor_series = donor_df.iloc[0]\n            st.warning("Donor has multiple samples; using the first.")\n\n        if set(recipient_df.columns) == set(donor_series.index):\n            recipient_df = recipient_df.reindex(columns=donor_series.index)\n\n        donor_vec = donor_series.to_numpy(dtype=float)\n        rec_names = list(recipient_df.index)\n        nb_list = []\n        prog = st.progress(0.0, text="Estimating bottleneck Nb per recipient ...")\n        for k, name in enumerate(rec_names):\n            x = recipient_df.loc[name].to_numpy()\n            nb_list.append(bottleneck_from_two_timepoints(\n                donor_vec, x, donor_pseudocount=pseudocount,\n                nb_max=nb_max, n_grid=n_grid, refine=refine))\n            prog.progress((k + 1) / len(rec_names))\n        prog.empty()\n        nb_arr = np.array(nb_list, dtype=float)\n\n        valid = np.isfinite(nb_arr)\n        if not valid.all():\n            st.warning(f"{int((~valid).sum())} recipient(s) had zero total counts and were dropped.")\n        recipient_used = recipient_df.loc[valid]\n        nb_used = nb_arr[valid]\n        if recipient_used.shape[0] == 0:\n            st.error("No usable recipients (all had zero counts).")\n            st.stop()\n\n        nb_table = pd.DataFrame({"recipient": list(recipient_used.index),\n                                 "Nb_hat": nb_used.astype(int)})\n\n        with st.spinner("Running neutrality LRT ..."):\n            results = neutrality_lrt(\n                donor_counts=donor_series,\n                recipient_counts=recipient_used,\n                bottleneck=nb_used,\n                pseudocount=pseudocount,\n                min_p=min_p,\n                fdr_alpha=fdr_alpha,\n                fdr_method=fdr_method,\n            )\n        st.session_state["results"] = results\n        st.session_state["nb_table"] = nb_table\n\n    if "results" in st.session_state:\n        results = st.session_state["results"]\n\n        if "nb_table" in st.session_state:\n            nb_table = st.session_state["nb_table"]\n            st.subheader("Estimated bottleneck (Nb) per recipient")\n            cA, cB = st.columns([3, 1])\n            with cA:\n                st.dataframe(nb_table, use_container_width=True, hide_index=True)\n            with cB:\n                st.metric("Mean Nb", f"{nb_table[\'Nb_hat\'].mean():.0f}")\n                st.metric("Median Nb", f"{nb_table[\'Nb_hat\'].median():.0f}")\n            st.download_button("Download Nb per recipient (CSV)",\n                               nb_table.to_csv(index=False).encode(),\n                               "bottleneck_Nb_per_recipient.csv", "text/csv")\n\n        n_tested   = results["pval"].notna().sum()\n        n_rejected = results["reject_FDR"].sum()\n        n_down_sig = int(((results["direction"] == "down_in_recipient") & results["reject_FDR"]).sum())\n        m1, m2, m3, m4 = st.columns(4)\n        m1.metric("Features tested", int(n_tested))\n        m2.metric(f"Rejected (FDR {int(fdr_alpha*100)}%)", int(n_rejected))\n        m3.metric("Not rejected (neutral)", int(n_tested - n_rejected))\n        m4.metric("down_in_recipient (sig.)", n_down_sig)\n\n        st.subheader("Results table")\n        display_cols = ["feature","p_donor","q_hat_recipient_center","LR","pval","qval_FDR","reject_FDR","direction"]\n        disp = results[display_cols].copy()\n        for c in ["p_donor","q_hat_recipient_center"]:\n            disp[c] = disp[c].map(lambda v: f"{v:.4e}" if pd.notna(v) else "")\n        disp["LR"] = disp["LR"].map(lambda v: f"{v:.3f}" if pd.notna(v) else "")\n        for c in ["pval","qval_FDR"]:\n            disp[c] = disp[c].map(lambda v: f"{v:.2e}" if pd.notna(v) else "")\n        st.dataframe(disp, use_container_width=True, hide_index=True)\n\n        down_df = results[(results["direction"] == "down_in_recipient") & (results["reject_FDR"])].copy()\n        dcol1, dcol2 = st.columns(2)\n        with dcol1:\n            st.download_button("Download full results CSV",\n                               results.to_csv(index=False).encode(),\n                               "neutrality_results.csv", "text/csv")\n        with dcol2:\n            st.download_button(f"Download significant down_in_recipient ({len(down_df)}) CSV",\n                               down_df.to_csv(index=False).encode(),\n                               "down_in_recipient_significant.csv", "text/csv")\n        st.caption("The down_in_recipient file lists only SIGNIFICANT depletions "\n                   "(reject_FDR = True and recipient frequency below donor frequency).")\n\n        st.subheader("log2 fold-change vs significance")\n        eps = 1e-12\n        plot_df = results.dropna(subset=["pval","q_hat_recipient_center","p_donor"]).copy()\n        plot_df["log2_fc"] = np.log2((plot_df["q_hat_recipient_center"] + eps) / (plot_df["p_donor"] + eps))\n        plot_df["neglog10_p"] = -np.log10(plot_df["pval"])\n        rej = plot_df[plot_df["reject_FDR"]]\n        hline = -np.log10(rej["pval"].max()) if len(rej) else -np.log10(fdr_alpha)\n        fig, ax = plt.subplots(figsize=(6.4, 5))\n        b = plot_df[~plot_df["reject_FDR"]]; r = plot_df[plot_df["reject_FDR"]]\n        ax.scatter(b["log2_fc"], b["neglog10_p"], s=14, c="#3b7fbf", alpha=0.75, edgecolors="none")\n        ax.scatter(r["log2_fc"], r["neglog10_p"], s=34, c="#e8261f", alpha=0.95, edgecolors="none")\n        ax.axvline(0.0, ls="--", lw=1, color="#6b7a99")\n        ax.axhline(hline, ls="--", lw=1, color="#6b7a99")\n        ax.set_xlabel(r"$log_2(\\hat{q}_j / D_j)$", fontsize=12)\n        ax.set_ylabel(r"$-log_{10}(p_{FDR})$", fontsize=12)\n        plt.tight_layout()\n        st.pyplot(fig, use_container_width=False)\n        st.caption(f"Red = rejected at FDR {fdr_alpha}. Vertical dashed line: no change. "\n                   f"Horizontal dashed line: FDR significance boundary. Left of centre = depleted in recipients.")\n        buf = io.BytesIO()\n        fig.savefig(buf, format="png", dpi=200, bbox_inches="tight")\n        st.download_button("Download scatter (PNG)", buf.getvalue(), "nonneutral_scatter.png", "image/png")\nelse:\n    st.info("Upload donor + recipient CSVs, or click Generate demo data to get started.")\n'
with open('app_neutrality.py', 'w', encoding='utf-8') as f:
    f.write(app_source)
print('app_neutrality.py' + ' written.')

app_neutrality.py written.


## 5 - Preview the app structure

In [5]:
import ast, pathlib
src = pathlib.Path('app_neutrality.py').read_text(encoding='utf-8')
tree = ast.parse(src)
names = []
for node in ast.walk(tree):
    if isinstance(node, ast.Call):
        nm = getattr(node.func, 'attr', None) or getattr(node.func, 'id', None)
        if nm in {'title','header','subheader','tabs','columns','button','file_uploader',
                  'slider','selectbox','metric','download_button','pyplot','progress','expander'}:
            names.append(f"  st.{nm}()")
print("App structure (Streamlit calls found):")
for s in dict.fromkeys(names):
    print(s)


App structure (Streamlit calls found):
  st.title()
  st.tabs()
  st.header()
  st.slider()
  st.selectbox()
  st.columns()
  st.button()
  st.expander()
  st.subheader()
  st.file_uploader()
  st.progress()
  st.metric()
  st.pyplot()
  st.download_button()


## 6 - Sanity-check: estimate Nb + run the test in-notebook

In [6]:
import importlib, sys
for m in ["neutrality_test","bottleneck_function"]:
    if m in sys.modules: del sys.modules[m]
from neutrality_test import neutrality_lrt
from bottleneck_function import bottleneck_from_two_timepoints
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
K, M = 40, 8
features = [f"Feature_{i:02d}" for i in range(K)]
samples  = [f"recipient_{i+1}" for i in range(M)]

donor_counts = rng.integers(50, 800, size=K).astype(float)
p = donor_counts / donor_counts.sum()
p_recip = p.copy()
p_recip[:6] *= rng.uniform(0.05, 0.3, size=6)
p_recip[6]  *= 3
p_recip[7]  *= 2
p_recip /= p_recip.sum()
recip_cols = {s: rng.multinomial(rng.integers(4000, 6000), p_recip) for s in samples}

# documented layout: rows = features, columns = samples
recipient_file_layout = pd.DataFrame(recip_cols, index=features)
donor_file_layout     = pd.DataFrame({"donor": donor_counts}, index=features)

# transpose as the app does
donor_series = donor_file_layout.T.iloc[0]
recipient_df = recipient_file_layout.T

# estimate Nb per recipient (bottleneck is NOT a parameter)
donor_vec = donor_series.to_numpy(dtype=float)
nb = [bottleneck_from_two_timepoints(donor_vec, recipient_df.loc[s].to_numpy()) for s in recipient_df.index]
print("Estimated Nb per recipient:", nb)

results = neutrality_lrt(donor_counts=donor_series, recipient_counts=recipient_df, bottleneck=np.array(nb, float))
print(results.head(8).to_string(index=False))
print("significant down_in_recipient:",
      int(((results["direction"]=="down_in_recipient") & results["reject_FDR"]).sum()))


Estimated Nb per recipient: [133, 136, 143, 146, 155, 131, 144, 134]
   feature  p_donor  q_hat_recipient_center         LR         pval     qval_FDR  reject_FDR         direction
Feature_05 0.038681                0.006869 100.177734 1.393169e-23 5.572676e-22        True down_in_recipient
Feature_01 0.035164                0.008664  63.732339 1.425256e-15 2.850512e-14        True down_in_recipient
Feature_07 0.031983                0.073108  36.394479 1.611590e-09 2.148787e-08        True   up_in_recipient
Feature_02 0.030141                0.012057  25.975001 3.458671e-07 3.458671e-06        True down_in_recipient
Feature_06 0.006363                0.023065  24.215595 8.613203e-07 6.890562e-06        True   up_in_recipient
Feature_04 0.020875                0.008335  19.773522 8.718209e-06 5.812140e-05        True down_in_recipient
Feature_03 0.021154                0.009328  16.076835 6.082351e-05 3.475629e-04        True down_in_recipient
Feature_32 0.006642                0.010578

## 7 - Launch the Streamlit app

Local URL: http://localhost:8501. The optional public link needs Node.js; if absent you'll see a note and can still use the local URL or deploy to Streamlit Community Cloud.

In [7]:
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "app_neutrality.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)
print("Streamlit started on http://localhost:8501")

try:
    subprocess.run(["npx", "--yes", "localtunnel", "--port", "8501"],
                   check=False, timeout=60)
except Exception as e:
    print(f"localtunnel not available ({e}). Use http://localhost:8501 locally, "
          "or deploy to Streamlit Community Cloud for a permanent public URL.")


Streamlit started on http://localhost:8501
localtunnel not available ([WinError 2] The system cannot find the file specified). Use http://localhost:8501 locally, or deploy to Streamlit Community Cloud for a permanent public URL.


---
### Deploy permanently

Push `app_neutrality.py`, `neutrality_test.py`, `bottleneck_function.py`, and `requirements.txt` to GitHub, then deploy at share.streamlit.io.

`requirements.txt`:
```
streamlit
numpy
pandas
scipy
statsmodels
matplotlib
```